In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random



In [2]:
def fetch_crypto_data_safe(target_records=2100):
    all_data = []
    page = 1
    per_page = 250

    print("Goal: Fetching at least {target_records} records")

    while len(all_data) < target_records:
        url = "https://api.coingecko.com/api/v3/coins/markets"
        params = {
            'vs_currency': 'usd',
            'order': 'market_cap_desc',
            'per_page': per_page,
            'page': page,
            'sparkline': 'false'}

        try:
            response = requests.get(url, params=params)


            if response.status_code == 429:
                wait_time = 30
                print(" Rate limit hit at page {page}. Sleeping for {wait_time}s")
                time.sleep(wait_time)
                continue

            if response.status_code != 200:
                print("Error {response.status_code} on page {page}. Stopping.")
                break

            data = response.json()
            if not data:
                break

            all_data.extend(data)
            print(f"Page {page} successful. Total: {len(all_data)}")

            page += 1
            time.sleep(5)

        except Exception as e:
            print(f"Connection error: {e}")
            time.sleep(10)

    return pd.DataFrame(all_data)



In [3]:
raw_df = fetch_crypto_data_safe(target_records=2200)

Goal: Fetching at least {target_records} records
Page 1 successful. Total: 250
Page 2 successful. Total: 500
Page 3 successful. Total: 750
 Rate limit hit at page {page}. Sleeping for {wait_time}s
 Rate limit hit at page {page}. Sleeping for {wait_time}s
Page 4 successful. Total: 1000
Page 5 successful. Total: 1250
Page 6 successful. Total: 1500
 Rate limit hit at page {page}. Sleeping for {wait_time}s
 Rate limit hit at page {page}. Sleeping for {wait_time}s
Page 7 successful. Total: 1750
Page 8 successful. Total: 2000
Page 9 successful. Total: 2250


In [4]:
cleaned_df = raw_df.drop_duplicates(subset=['id']).dropna(subset=['current_price'])
cleaned_df = cleaned_df[['name', 'symbol', 'current_price', 'market_cap', 'total_volume']]

In [5]:
print(cleaned_df)

              name   symbol  current_price     market_cap  total_volume
0          Bitcoin      btc   90482.000000  1806857453346  2.851830e+10
1         Ethereum      eth    3089.130000   372658294176  1.298239e+10
2           Tether     usdt       0.998756   186737282793  4.749355e+10
3              XRP      xrp       2.090000   126721710144  2.288538e+09
4              BNB      bnb     903.870000   124497576713  1.220076e+09
...            ...      ...            ...            ...           ...
2245  Hemi Bitcoin  hemibtc   90087.000000        4487938  4.430000e+04
2246   JETT CRYPTO     jett       0.083130        4470122  9.207800e+04
2247        Xterio     xter       0.031351        4469295  2.285685e+06
2248      LumiWave      lwa       0.005795        4462814  4.439040e+05
2249    GameFi.org     gafi       0.406857        4453041  1.089480e+05

[2250 rows x 5 columns]


In [6]:
print("Finished! Final record count: {len(cleaned_df)}")
cleaned_df.to_csv("crypto_2000_records.csv", index=False)

Finished! Final record count: {len(cleaned_df)}
